# Capstone Project — Production-Grade Multi-Account AWS Environment

**Level: Intermediate–Advanced**

## Objective

You are the lead **CloudOps Engineer and Solutions Architect** for a fintech startup called **FinOpsCloud**. Your task is to design, deploy, and operationalize a production-grade AWS environment from scratch.

### What You'll Build
A complete, well-architected AWS environment including:
1. Multi-account structure with AWS Organizations
2. Production 3-tier web application (ALB → ECS Fargate → Aurora)
3. Full observability stack (CloudWatch, CloudTrail, Security Hub)
4. Security controls (KMS, Secrets Manager, WAF, GuardDuty)
5. Disaster recovery plan (pilot light pattern)
6. CI/CD pipeline with blue/green deployments
7. Cost governance (tagging, budgets, Savings Plans analysis)

### Rules
- Work through steps in order — each step builds on the previous
- Write and test each CloudFormation template before moving on
- Run the verification checklist after each section
- Stretch goals are optional — tackle them after core project is complete

**Estimated time:** 8–12 hours

_References: Apply lessons from all 9 phases of this course._

## Prerequisites

Before starting, confirm you have:
- [ ] AWS account with programmatic access (`aws configure`)
- [ ] AWS CLI v2 installed (`aws --version`)
- [ ] Python 3.10+ with boto3 (`pip install boto3`)
- [ ] Completed Phases 1–9 of this course (or strong AWS experience)
- [ ] At least $10 budget (most resources are free-tier eligible; cleanup cells provided)

**Architecture Diagram:**
```
Internet → Route 53 → CloudFront → WAF → ALB (public subnets)
                                           ↓
                                   ECS Fargate (private subnets)
                                           ↓
                                   Aurora PostgreSQL (database subnets)
                                           ↓
                          ElastiCache Redis (session/cache)

Observability: CloudWatch + CloudTrail + Security Hub + GuardDuty
Secrets: KMS + Secrets Manager
IaC: CloudFormation (all resources)
CI/CD: CodePipeline + CodeBuild + CodeDeploy (blue/green on ECS)
```

In [ ]:
import boto3, json, subprocess, os
from datetime import datetime

# Verify prerequisites
account_id = boto3.client('sts').get_caller_identity()['Account']
region = boto3.session.Session().region_name or 'us-east-1'
print(f'Account: {account_id}')
print(f'Region: {region}')

# Project configuration
PROJECT = 'finopscloud'
ENV = 'prod'
print(f'Project: {PROJECT}')
print(f'Environment: {ENV}')

## Step 1: Foundation — VPC & Network Architecture

**File:** `01-vpc.yaml`
**Lesson reference:** Lesson 6.1 (VPC Fundamentals)

Create a production VPC with:
- 3 AZs for high availability
- Public, private, and database subnet tiers
- Internet Gateway and NAT Gateways (one per AZ)
- VPC Flow Logs to CloudWatch
- S3 Gateway Endpoint (free, route S3 traffic privately)

In [ ]:
%%writefile 01-vpc.yaml
AWSTemplateFormatVersion: '2010-09-09'
Description: 'FinOpsCloud - Production VPC with 3-tier architecture'

Parameters:
  ProjectName:
    Type: String
    Default: finopscloud
  Environment:
    Type: String
    Default: prod

Resources:
  VPC:
    Type: AWS::EC2::VPC
    Properties:
      CidrBlock: 10.0.0.0/16
      EnableDnsSupport: true
      EnableDnsHostnames: true
      Tags:
        - Key: Name
          Value: !Sub '${ProjectName}-${Environment}-vpc'
        - Key: Environment
          Value: !Ref Environment

  InternetGateway:
    Type: AWS::EC2::InternetGateway
    Properties:
      Tags:
        - Key: Name
          Value: !Sub '${ProjectName}-${Environment}-igw'

  VPCGatewayAttachment:
    Type: AWS::EC2::VPCGatewayAttachment
    Properties:
      VpcId: !Ref VPC
      InternetGatewayId: !Ref InternetGateway

  # Public Subnets
  PublicSubnet1:
    Type: AWS::EC2::Subnet
    Properties:
      VpcId: !Ref VPC
      CidrBlock: 10.0.0.0/24
      AvailabilityZone: !Select [0, !GetAZs '']
      MapPublicIpOnLaunch: true
      Tags:
        - Key: Name
          Value: !Sub '${ProjectName}-${Environment}-public-1'

  PublicSubnet2:
    Type: AWS::EC2::Subnet
    Properties:
      VpcId: !Ref VPC
      CidrBlock: 10.0.1.0/24
      AvailabilityZone: !Select [1, !GetAZs '']
      MapPublicIpOnLaunch: true
      Tags:
        - Key: Name
          Value: !Sub '${ProjectName}-${Environment}-public-2'

  PublicSubnet3:
    Type: AWS::EC2::Subnet
    Properties:
      VpcId: !Ref VPC
      CidrBlock: 10.0.2.0/24
      AvailabilityZone: !Select [2, !GetAZs '']
      MapPublicIpOnLaunch: true
      Tags:
        - Key: Name
          Value: !Sub '${ProjectName}-${Environment}-public-3'

  # Private App Subnets
  PrivateSubnet1:
    Type: AWS::EC2::Subnet
    Properties:
      VpcId: !Ref VPC
      CidrBlock: 10.0.10.0/24
      AvailabilityZone: !Select [0, !GetAZs '']
      Tags:
        - Key: Name
          Value: !Sub '${ProjectName}-${Environment}-private-1'

  PrivateSubnet2:
    Type: AWS::EC2::Subnet
    Properties:
      VpcId: !Ref VPC
      CidrBlock: 10.0.11.0/24
      AvailabilityZone: !Select [1, !GetAZs '']
      Tags:
        - Key: Name
          Value: !Sub '${ProjectName}-${Environment}-private-2'

  PrivateSubnet3:
    Type: AWS::EC2::Subnet
    Properties:
      VpcId: !Ref VPC
      CidrBlock: 10.0.12.0/24
      AvailabilityZone: !Select [2, !GetAZs '']
      Tags:
        - Key: Name
          Value: !Sub '${ProjectName}-${Environment}-private-3'

  # Database Subnets
  DatabaseSubnet1:
    Type: AWS::EC2::Subnet
    Properties:
      VpcId: !Ref VPC
      CidrBlock: 10.0.20.0/27
      AvailabilityZone: !Select [0, !GetAZs '']
      Tags:
        - Key: Name
          Value: !Sub '${ProjectName}-${Environment}-database-1'

  DatabaseSubnet2:
    Type: AWS::EC2::Subnet
    Properties:
      VpcId: !Ref VPC
      CidrBlock: 10.0.20.32/27
      AvailabilityZone: !Select [1, !GetAZs '']
      Tags:
        - Key: Name
          Value: !Sub '${ProjectName}-${Environment}-database-2'

  # NAT Gateway (AZ-1 for demo; add NAT2/3 for full HA)
  EIP1:
    Type: AWS::EC2::EIP
    Properties:
      Domain: vpc

  NATGateway1:
    Type: AWS::EC2::NatGateway
    Properties:
      AllocationId: !GetAtt EIP1.AllocationId
      SubnetId: !Ref PublicSubnet1
      Tags:
        - Key: Name
          Value: !Sub '${ProjectName}-${Environment}-nat-1'

  # Route Tables
  PublicRouteTable:
    Type: AWS::EC2::RouteTable
    Properties:
      VpcId: !Ref VPC

  PublicRoute:
    Type: AWS::EC2::Route
    Properties:
      RouteTableId: !Ref PublicRouteTable
      DestinationCidrBlock: 0.0.0.0/0
      GatewayId: !Ref InternetGateway

  PrivateRouteTable:
    Type: AWS::EC2::RouteTable
    Properties:
      VpcId: !Ref VPC

  PrivateRoute:
    Type: AWS::EC2::Route
    Properties:
      RouteTableId: !Ref PrivateRouteTable
      DestinationCidrBlock: 0.0.0.0/0
      NatGatewayId: !Ref NATGateway1

  # Subnet Associations
  PubSub1Association: {Type: AWS::EC2::SubnetRouteTableAssociation, Properties: {SubnetId: !Ref PublicSubnet1, RouteTableId: !Ref PublicRouteTable}}
  PubSub2Association: {Type: AWS::EC2::SubnetRouteTableAssociation, Properties: {SubnetId: !Ref PublicSubnet2, RouteTableId: !Ref PublicRouteTable}}
  PubSub3Association: {Type: AWS::EC2::SubnetRouteTableAssociation, Properties: {SubnetId: !Ref PublicSubnet3, RouteTableId: !Ref PublicRouteTable}}
  PrivSub1Association: {Type: AWS::EC2::SubnetRouteTableAssociation, Properties: {SubnetId: !Ref PrivateSubnet1, RouteTableId: !Ref PrivateRouteTable}}
  PrivSub2Association: {Type: AWS::EC2::SubnetRouteTableAssociation, Properties: {SubnetId: !Ref PrivateSubnet2, RouteTableId: !Ref PrivateRouteTable}}
  PrivSub3Association: {Type: AWS::EC2::SubnetRouteTableAssociation, Properties: {SubnetId: !Ref PrivateSubnet3, RouteTableId: !Ref PrivateRouteTable}}

  # S3 Gateway Endpoint (free, traffic stays on AWS)
  S3Endpoint:
    Type: AWS::EC2::VPCEndpoint
    Properties:
      VpcId: !Ref VPC
      ServiceName: !Sub 'com.amazonaws.${AWS::Region}.s3'
      VpcEndpointType: Gateway
      RouteTableIds:
        - !Ref PublicRouteTable
        - !Ref PrivateRouteTable

Outputs:
  VpcId: {Value: !Ref VPC, Export: {Name: !Sub '${AWS::StackName}-VpcId'}}
  PublicSubnet1Id: {Value: !Ref PublicSubnet1, Export: {Name: !Sub '${AWS::StackName}-PublicSubnet1'}}
  PublicSubnet2Id: {Value: !Ref PublicSubnet2, Export: {Name: !Sub '${AWS::StackName}-PublicSubnet2'}}
  PublicSubnet3Id: {Value: !Ref PublicSubnet3, Export: {Name: !Sub '${AWS::StackName}-PublicSubnet3'}}
  PrivateSubnet1Id: {Value: !Ref PrivateSubnet1, Export: {Name: !Sub '${AWS::StackName}-PrivateSubnet1'}}
  PrivateSubnet2Id: {Value: !Ref PrivateSubnet2, Export: {Name: !Sub '${AWS::StackName}-PrivateSubnet2'}}
  PrivateSubnet3Id: {Value: !Ref PrivateSubnet3, Export: {Name: !Sub '${AWS::StackName}-PrivateSubnet3'}}
  DatabaseSubnet1Id: {Value: !Ref DatabaseSubnet1, Export: {Name: !Sub '${AWS::StackName}-DatabaseSubnet1'}}
  DatabaseSubnet2Id: {Value: !Ref DatabaseSubnet2, Export: {Name: !Sub '${AWS::StackName}-DatabaseSubnet2'}}

In [ ]:
# Validate and deploy VPC stack
import subprocess, json, boto3, time

cf = boto3.client('cloudformation', region_name='us-east-1')
STACK_NAME = 'finopscloud-vpc'

# Validate
result = subprocess.run('aws cloudformation validate-template --template-body file://01-vpc.yaml', 
                        shell=True, capture_output=True, text=True)
if result.returncode == 0:
    print('Template VALID')
else:
    print(f'Template INVALID: {result.stderr}')

# Deploy using change sets
with open('01-vpc.yaml') as f:
    template = f.read()

try:
    cf.describe_stacks(StackName=STACK_NAME)
    changeset_type = 'UPDATE'
except:
    changeset_type = 'CREATE'

print(f'Deploying {STACK_NAME} ({changeset_type})...')

# TODO: Create and execute change set (follow pattern from Lesson 1.4)
# YOUR CODE HERE

## Step 2: Security Foundation — KMS, IAM Roles & Security Groups

**Lesson reference:** Lessons 5.1 & 5.2 (IAM, KMS)

Create:
- KMS Customer Managed Key for data encryption
- IAM roles: ECS task execution role, CI/CD pipeline role, monitoring role
- Security groups: ALB, ECS tasks, Aurora RDS, ElastiCache

In [ ]:
# TODO: Create security infrastructure
# 1. Create KMS CMK for the project with key rotation enabled
# 2. Create security groups:
#    - alb-sg: Allow inbound 80, 443 from 0.0.0.0/0
#    - ecs-sg: Allow inbound 3000 from alb-sg only
#    - rds-sg: Allow inbound 5432 from ecs-sg only
#    - cache-sg: Allow inbound 6379 from ecs-sg only
# 3. Create IAM roles:
#    - ecs-task-execution-role (AmazonECSTaskExecutionRolePolicy + access to Secrets Manager)
#    - ecs-task-role (application permissions: S3, SQS, CloudWatch)

import boto3, json

kms = boto3.client('kms', region_name='us-east-1')
iam = boto3.client('iam')
ec2 = boto3.client('ec2', region_name='us-east-1')

account_id = boto3.client('sts').get_caller_identity()['Account']

# TODO: Implement all three components above
print('Step 2: TODO - Implement security foundation')

## Step 3: Database Layer — Aurora PostgreSQL + ElastiCache

**Lesson reference:** Lessons 3.3, 3.4, 8.2 (DR, Backup, Performance)

Create:
- Aurora PostgreSQL cluster (Multi-AZ, encrypted with CMK)
- DB credentials stored in Secrets Manager
- ElastiCache Redis replication group (Multi-AZ)
- DB subnet group using database subnets

In [ ]:
# TODO: Create database infrastructure
# 1. Store Aurora credentials in Secrets Manager
# 2. Create RDS subnet group (using DatabaseSubnet1 and DatabaseSubnet2)
# 3. Create Aurora PostgreSQL cluster:
#    - Engine: aurora-postgresql, version: 15.x
#    - Multi-AZ: True
#    - Encrypted: True (using CMK)
#    - BackupRetentionPeriod: 7
#    - Deletion protection: True (prod best practice)
# 4. Create ElastiCache Redis replication group:
#    - 2 nodes (1 primary + 1 replica)
#    - Encrypted at rest and in transit

import boto3
print('Step 3: TODO - Implement database layer')

## Step 4: Application Layer — ECS Fargate + ALB

**Lesson reference:** Lessons 3.1, 8.3 (ASG/ELB, Serverless/Containers)

Create:
- ECR repository for container images
- ECS cluster with Fargate
- ECS Task Definition (referencing Secrets Manager for DB creds)
- ECS Service (desired: 2, spread across AZs)
- Application Load Balancer with target group and health check
- ALB listener rules

In [ ]:
# TODO: Deploy application layer
# 1. Create ECR repository: finopscloud-api
# 2. Create ECS cluster (Fargate)
# 3. Create task definition:
#    - Container: finopscloud-api:latest (from ECR)
#    - CPU: 256, Memory: 512
#    - Secrets: DB_HOST, DB_PASSWORD from Secrets Manager
#    - LogDriver: awslogs (to CloudWatch Logs)
# 4. Create ECS Service:
#    - LaunchType: FARGATE
#    - DesiredCount: 2
#    - Spread across AZs
#    - DeploymentConfiguration: blue/green (CodeDeploy)
# 5. Create ALB in public subnets
# 6. Create target group and listener

import boto3
print('Step 4: TODO - Deploy application layer')

## Step 5: Observability — CloudWatch, CloudTrail & Alarms

**Lesson reference:** Lessons 2.1–2.4 (Monitoring, Logging)

Create:
- CloudTrail multi-region trail → S3 bucket
- CloudWatch Alarms: ALB 5xx errors, ECS CPU, RDS CPU, Aurora storage
- CloudWatch Dashboard: operations overview
- SNS topic for operational alerts
- Log groups with 30-day retention for ECS container logs

In [ ]:
# TODO: Build observability stack
# 1. Create SNS topic: finopscloud-ops-alerts
# 2. Create CloudWatch Alarms:
#    a. ALB 5xx rate > 5% for 2 minutes
#    b. ECS CPU > 80% for 5 minutes
#    c. RDS CPU > 90% for 5 minutes
#    d. RDS free storage < 10GB
# 3. Create composite alarm: ALARM if (5xx_high AND ecs_cpu_high)
# 4. Create CloudWatch Dashboard with widgets for all key metrics
# 5. Create CloudTrail multi-region trail

import boto3
print('Step 5: TODO - Build observability stack')

## Step 6: Security Controls — WAF, GuardDuty & Config

**Lesson reference:** Lessons 5.3, 6.4 (Compliance, CloudFront/WAF)

Enable:
- AWS WAF on the ALB (managed rules: Core Rule Set + IP Reputation)
- Amazon GuardDuty
- AWS Config with key rules: encrypted-volumes, restricted-ssh, s3-public-block
- Security Hub (enable AWS Foundational Security Best Practices)

In [ ]:
# TODO: Configure security controls
# 1. Enable GuardDuty
# 2. Create WAF WebACL with managed rules and rate limiting
# 3. Associate WAF with ALB
# 4. Enable AWS Config with conformance pack (AWS-FoundationalSecurityBestPractices)
# 5. Enable Security Hub

import boto3
print('Step 6: TODO - Configure security controls')

## Step 7: CI/CD Pipeline — CodePipeline with Blue/Green

**Lesson reference:** Lessons 4.3, 4.5 (Deployment Strategies, Event-Driven Automation)

Create:
- CodeCommit (or GitHub) source repository
- CodeBuild project: build Docker image, push to ECR
- CodeDeploy application: ECS blue/green deployment
- CodePipeline: Source → Build → Deploy (Blue/Green)
- Rollback trigger: CloudWatch Alarm on ALB 5xx rate

In [ ]:
# TODO: Build CI/CD pipeline
# 1. Create CodeCommit repository: finopscloud-api
# 2. Create CodeBuild project:
#    - Build Docker image
#    - Tag with git commit hash
#    - Push to ECR
#    - Run unit tests
# 3. Create CodeDeploy ECS deployment group (Blue/Green)
#    - With CloudWatch alarm rollback on 5xx
# 4. Create CodePipeline: Source → Build → Deploy
# 5. Test by committing a change and watching the pipeline execute

import boto3
print('Step 7: TODO - Build CI/CD pipeline')

## Step 8: Cost Governance — Tagging, Budgets & Savings Plan Analysis

**Lesson reference:** Lesson 7.4 (Cost Optimization)

Configure:
- Cost allocation tag activation (Environment, Project, Team)
- AWS Budget: alert at 80% of $200/month
- Trusted Advisor check for idle/underutilized resources
- Cost Explorer Savings Plans recommendations

In [ ]:
# TODO: Configure cost governance
# 1. Create AWS Budget: $200/month with 80% alert
# 2. Check Cost Explorer for Savings Plans recommendations
# 3. Run a cost leak audit (from Lesson 7.4 exercise)

import boto3
print('Step 8: TODO - Configure cost governance')

## Verification Checklist

Run this cell to automatically verify your environment:

In [ ]:
import boto3, json

ec2 = boto3.client('ec2', region_name='us-east-1')
rds = boto3.client('rds', region_name='us-east-1')
elbv2 = boto3.client('elbv2', region_name='us-east-1')
ecs = boto3.client('ecs', region_name='us-east-1')
guardduty = boto3.client('guardduty', region_name='us-east-1')
cloudtrail = boto3.client('cloudtrail', region_name='us-east-1')

checks = []

def check(name, condition, detail=''):
    status = '✓' if condition else '✗'
    checks.append({'name': name, 'passed': condition, 'detail': detail})
    print(f'{status} {name}: {detail}')

print('=== FinOpsCloud Capstone Verification ===')
print()

# 1. VPC exists and has subnets in 3 AZs
try:
    vpcs = ec2.describe_vpcs(Filters=[{'Name': 'tag:Name', 'Values': ['finopscloud-*']}])['Vpcs']
    check('VPC created', len(vpcs) > 0, f'{len(vpcs)} VPC(s) found')
    if vpcs:
        subnets = ec2.describe_subnets(Filters=[{'Name': 'vpc-id', 'Values': [vpcs[0]['VpcId']]}])['Subnets']
        azs = len(set(s['AvailabilityZone'] for s in subnets))
        check('Multi-AZ subnets', azs >= 2, f'{len(subnets)} subnets across {azs} AZs')
except Exception as e:
    check('VPC created', False, str(e))

# 2. ALB exists
try:
    lbs = elbv2.describe_load_balancers()['LoadBalancers']
    finops_lbs = [lb for lb in lbs if 'finopscloud' in lb['LoadBalancerName'].lower()]
    check('ALB created', len(finops_lbs) > 0, f'{len(finops_lbs)} LB(s)')
except Exception as e:
    check('ALB created', False, str(e))

# 3. ECS Cluster exists with running tasks
try:
    clusters = ecs.list_clusters()['clusterArns']
    finops_clusters = [c for c in clusters if 'finopscloud' in c.lower()]
    check('ECS Cluster created', len(finops_clusters) > 0, f'{len(finops_clusters)} cluster(s)')
except Exception as e:
    check('ECS Cluster created', False, str(e))

# 4. RDS Multi-AZ
try:
    dbs = rds.describe_db_instances()['DBInstances']
    finops_dbs = [db for db in dbs if 'finopscloud' in db['DBInstanceIdentifier'].lower()]
    if finops_dbs:
        multi_az = finops_dbs[0]['MultiAZ']
        encrypted = finops_dbs[0]['StorageEncrypted']
        check('RDS Multi-AZ', multi_az, 'Multi-AZ enabled' if multi_az else 'NOT Multi-AZ!')
        check('RDS Encrypted', encrypted, 'Encrypted at rest' if encrypted else 'NOT encrypted!')
    else:
        check('RDS created', False, 'No finopscloud RDS instance found')
except Exception as e:
    check('RDS', False, str(e))

# 5. GuardDuty enabled
try:
    detectors = guardduty.list_detectors()['DetectorIds']
    check('GuardDuty enabled', len(detectors) > 0, f'{len(detectors)} detector(s)')
except Exception as e:
    check('GuardDuty enabled', False, str(e))

# 6. CloudTrail
try:
    trails = cloudtrail.describe_trails()['trailList']
    multi_region = [t for t in trails if t.get('IsMultiRegionTrail')]
    check('CloudTrail multi-region', len(multi_region) > 0, f'{len(multi_region)} multi-region trail(s)')
except Exception as e:
    check('CloudTrail', False, str(e))

# Summary
passed = sum(1 for c in checks if c['passed'])
total = len(checks)
print()
print(f'=== Results: {passed}/{total} checks passed ===')
if passed == total:
    print('CONGRATULATIONS! Your production-grade AWS environment is complete!')
else:
    print(f'{total - passed} check(s) to complete. Review the output above for failed items.')

## Stretch Goals

Complete these after the core project to deepen your expertise:

1. **Multi-Region DR** (Lesson 3.3): Implement pilot light DR in us-west-2
   - Aurora Global Database for cross-region replication
   - Route 53 failover routing with health checks
   
2. **CloudFront Distribution** (Lesson 6.4): Add CDN in front of ALB
   - Cache static assets with long TTL
   - Move WAF to CloudFront (global edge protection)

3. **Multi-Account** (Lesson 7.1): Create dev and staging accounts in AWS Organizations
   - Deploy same CloudFormation to each account using StackSets
   - Add SCPs to prevent accidental resource creation in prod

4. **Performance Optimization** (Lesson 8.2): Add ElastiCache DAX in front of DynamoDB
   - Add read replica to Aurora for reporting workloads
   - Implement connection pooling with RDS Proxy

5. **Serverless Modernization** (Lesson 9.4): Refactor the batch reporting endpoint to Lambda
   - API Gateway → Lambda → Aurora (read-only)
   - Step Functions for complex report generation workflow

In [ ]:
# CLEANUP — Run this to tear down all capstone resources and avoid ongoing charges
# WARNING: This will delete all resources created in this capstone

import boto3

print('Capstone Resource Cleanup')
print('=========================')
print()
print('IMPORTANT: Delete resources in this order to avoid dependency errors:')
print('1. CodePipeline / CodeDeploy')
print('2. ECS Service (scale to 0 first, then delete)')
print('3. ALB + Target Groups')
print('4. ElastiCache cluster')
print('5. Aurora cluster (disable deletion protection first!)')
print('6. ECR repository')
print('7. CloudFormation stacks (in reverse order)')
print('8. KMS scheduled key deletion')
print('9. CloudTrail trail + S3 bucket (empty bucket first)')
print()
print('To delete all CloudFormation stacks:')
print('aws cloudformation delete-stack --stack-name finopscloud-vpc')
print()
print('Always verify in the AWS Console that resources are fully deleted before stopping.')